In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv("SampleSuperstore.csv")
df = df.drop_duplicates().drop(columns=['Country'])
df['Postal Code'] = df['Postal Code'].astype(str).str.zfill(5)
df.columns = [c.replace(' ', '_').replace('-', '_') for c in df.columns]

con = sqlite3.connect("superstore.db")
df.to_sql("orders", con, index=False, if_exists="replace")

def q(sql):
    return pd.read_sql(sql, con)

q("SELECT * FROM orders LIMIT 5")

,Ship_Mode,Segment,City,State,Postal_Code,Region,Category,Sub_Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,Second Class,Consumer,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,Second Class,Corporate,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,Standard Class,Consumer,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,Standard Class,Consumer,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164


Q1: Overall KPIs (SELECT, aggregate functions)

In [2]:
q("""
SELECT COUNT(*) AS total_rows,
       ROUND(SUM(Sales))  AS total_sales,
       ROUND(SUM(Profit)) AS total_profit,
       ROUND(SUM(Profit) * 100.0 / SUM(Sales), 1) AS margin_pct
FROM orders
""")

,total_rows,total_sales,total_profit,margin_pct
0,9977,2296196.0,286241.0,12.5


Q2: Profit by region (GROUP BY, ORDER BY)

In [4]:
q("""
SELECT Region, ROUND(SUM(Sales)) AS sales, ROUND(SUM(Profit)) AS profit
FROM orders
GROUP BY Region
ORDER BY profit DESC
""")

,Region,sales,profit
0,West,725256.0,108330.0
1,East,678435.0,91506.0
2,South,391722.0,46749.0
3,Central,500783.0,39656.0


Q3: Which sub-categories lose money?

In [5]:
q("""
SELECT Sub_Category, ROUND(SUM(Profit)) AS profit
FROM orders
GROUP BY Sub_Category
HAVING SUM(Profit) < 0
ORDER BY profit
""")

,Sub_Category,profit
0,Tables,-17725.0
1,Bookcases,-3473.0
2,Supplies,-1189.0


Q4: Discount bands (CASE WHEN)

In [6]:
q("""
SELECT CASE WHEN Discount = 0   THEN '0%'
            WHEN Discount <= 0.2 THEN '1-20%'
            WHEN Discount <= 0.4 THEN '21-40%'
            ELSE '>40%' END AS discount_band,
       COUNT(*) AS order_lines,
       ROUND(AVG(Profit), 1) AS avg_profit,
       ROUND(SUM(Profit)) AS total_profit
FROM orders
GROUP BY discount_band
ORDER BY MIN(Discount)
""")

,discount_band,order_lines,avg_profit,total_profit
0,0%,4787,67.0,320844.0
1,1-20%,3799,26.5,100755.0
2,21-40%,459,-78.0,-35805.0
3,>40%,932,-106.8,-99552.0


Q5: Profit share by category (CTE, a named temporary result)

In [7]:
q("""
WITH cat AS (
    SELECT Category, SUM(Sales) AS s, SUM(Profit) AS p
    FROM orders
    GROUP BY Category
)
SELECT Category,
       ROUND(s) AS sales,
       ROUND(p) AS profit,
       ROUND(p * 100.0 / s, 1) AS margin_pct,
       ROUND(p * 100.0 / (SELECT SUM(p) FROM cat), 1) AS share_of_profit
FROM cat
ORDER BY p DESC
""")

,Category,sales,profit,margin_pct,share_of_profit
0,Technology,836154.0,145455.0,17.4,50.8
1,Office Supplies,718735.0,122365.0,17.0,42.7
2,Furniture,741306.0,18422.0,2.5,6.4


Q6: Rank sub-categories inside each category (window function)

In [8]:
q("""
SELECT Category, Sub_Category,
       ROUND(SUM(Profit)) AS profit,
       RANK() OVER (PARTITION BY Category ORDER BY SUM(Profit) DESC) AS rnk
FROM orders
GROUP BY Category, Sub_Category
ORDER BY Category, rnk
""")

,Category,Sub_Category,profit,rnk
0,Furniture,Chairs,26567.0,1
1,Furniture,Furnishings,13053.0,2
2,Furniture,Bookcases,-3473.0,3
3,Furniture,Tables,-17725.0,4
4,Office Supplies,Paper,33944.0,1
5,Office Supplies,Binders,30228.0,2
6,Office Supplies,Storage,21279.0,3
7,Office Supplies,Appliances,18138.0,4
8,Office Supplies,Envelopes,6964.0,5
9,Office Supplies,Art,6525.0,6


Q7: Two worst states per region (ROW_NUMBER)

In [9]:
q("""
WITH s AS (
    SELECT Region, State, SUM(Profit) AS p
    FROM orders
    GROUP BY Region, State
),
r AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY Region ORDER BY p ASC) AS rn
    FROM s
)
SELECT Region, State, ROUND(p) AS profit
FROM r
WHERE rn <= 2
ORDER BY Region, rn
""")

,Region,State,profit
0,Central,Texas,-25751.0
1,Central,Illinois,-12602.0
2,East,Ohio,-16959.0
3,East,Pennsylvania,-15565.0
4,South,North Carolina,-7491.0
5,South,Tennessee,-5342.0
6,West,Colorado,-6528.0
7,West,Arizona,-3428.0
